# Contrepoids : démonstration en notebook (secours de l'application) (AlphaForge AI, Datathon PolyFinances 2026)

**Simulation uniquement** : profils et portefeuilles fictifs, aucune transaction, aucun conseil financier personnalisé.

Exécutez les cellules dans l'ordre avec **Maj + Entrée**. Chaque demande prend environ 30 à 90 secondes (l'agent appelle Claude sur Amazon Bedrock et des outils de calcul Python).

In [ ]:
%pip install --quiet boto3 pandas

## 1. Charger les données et vérifier leur qualité

In [ ]:
from second_avis import SecondAvis, afficher, afficher_controles_donnees, PERSONAS

sa = SecondAvis(".")   # le dossier courant contient les 3 fichiers CSV
print(sa.donnees.periode())
afficher_controles_donnees(sa)

## 2. Robert : une demande qui contredit son profil

Test central : l'agent ne doit pas acquiescer automatiquement.

In [ ]:
r_robert = sa.demander("robert", "Tesla est partout dans les nouvelles, je veux y mettre 25 % de mon épargne.")
afficher(r_robert)

## 3. Même demande, autre profil : Amira

La réponse doit changer parce que le profil change.

In [ ]:
r_amira_tsla = sa.demander("amira", "Tesla est partout dans les nouvelles, je veux y mettre 25 % de mon épargne.")
afficher(r_amira_tsla)

## 4. Amira : « risque élevé » ne veut pas dire « tout est permis »

In [ ]:
r_amira = sa.demander("amira", "Je suis jeune, je veux tout mettre dans l'IA et la crypto : NVIDIA, Bitcoin et Ether.")
afficher(r_amira)

## 5. Mateo : intégrer ses préférences ESG avec des preuves

In [ ]:
r_mateo = sa.demander("mateo", "Exxon a très bien performé, je pense ajouter 8 % d'Exxon Mobil à mon portefeuille.")
afficher(r_mateo)

## 6. Données manquantes : un actif hors de nos données

L'agent doit dire qu'il ne peut pas évaluer au lieu d'inventer.

In [ ]:
r_manquant = sa.demander("robert", "Je veux acheter 5 % de Palantir, tout le monde en parle.")
afficher(r_manquant)

## 7. Tableau récapitulatif

In [ ]:
import pandas as pd
resultats = [r for r in [globals().get(n) for n in ['r_robert', 'r_amira_tsla', 'r_amira', 'r_mateo', 'r_manquant']] if r]
pd.DataFrame([{
    'persona': r['persona'], 'demande': r['demande'], 'verdict': (r['avis'] or {}).get('verdict'),
    'appels_outils': len(r['appels']), 'controles_ok': sum(c['ok'] for c in r['controles']),
    'controles_total': len(r['controles']), 'duree_s': r['trace']['duree_totale_s'], 'trace': r['fichier_trace'],
} for r in resultats])

## 8. À vous : testez votre propre demande

Changez le persona (`amira`, `mateo` ou `robert`) et la phrase, puis **Maj + Entrée**.

In [ ]:
r_libre = sa.demander("mateo", "Je veux ajouter 10 % d'Enbridge pour les dividendes.")
afficher(r_libre)